# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()
print("hey")

hey


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "latest") \
    .load()
print("hey")

hey


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")
print("hey")

hey


In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )
print("hey")

hey


In [6]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()
print("hey")

hey


### Demo output

Counting edits by bot / human in 1-hour windows. In "complete" output mode, Spark re-prints the full state at every batch, so the counters of the current window keep growing until the hour ends.

```
-------------------------------------------
Batch: 5
-------------------------------------------
|window_start       |window_end         |bot  |edit_count|
|2026-10-01 14:00:00|2026-10-01 15:00:00|false|12        |
|2026-10-01 14:00:00|2026-10-01 15:00:00|true |3         |
```

A batch is not a slice of the window: it is one processing round of Spark (every few seconds). A window is a slice of the event time. With a 1-hour window only two rows are shown for a whole hour, which is why the explorations below use much shorter windows.

In [7]:
# stop the demo query to keep the console readable
streaming_query_df.stop()
print("hey")

hey


## Q12 - Tumbling windows vs Overlapping windows

We first build a common dataframe with a proper timestamp column (required by the watermark) and the size of each edit:

 "event_time": the event timestamp cast to a real timestamp type
 "size_change" = length.new - length.old`: bytes added (positive) or removed (negative) by the edit

A watermark is added to every query below so Spark can close and drop old windows instead of keeping them in memory forever.

In [8]:
from pyspark.sql.functions import to_timestamp, avg, sum as _sum, abs as _abs

# base dataframe with a proper event time column and the size of each edit
events_df = parsed_df \
    .withColumn("event_time", to_timestamp(from_unixtime(col("timestamp")))) \
    .withColumn("size_change", col("length.new") - col("length.old"))
print("hey")

hey


### Exploration 1 - Tumbling windows (30 seconds)

"window(col("event_time"), "30 seconds")" with a single duration creates fixed slices that never overlap: each edit belongs to exactly one window. Grouped by "bot", with the edit count, the total bytes and the average bytes per window.

In [27]:
tumbling_df = events_df \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        window(col("event_time"), "30 seconds"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        _sum("size_change").alias("total_bytes"),
        avg("size_change").alias("avg_bytes")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
        col("total_bytes"),
        col("avg_bytes")
    )

tumbling_query = tumbling_df \
    .writeStream \
    .queryName("tumbling_30s") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()
print("hey")

hey


**Output (French Wikipedia)**

```
-------------------------------------------
Batch: 7
-------------------------------------------
|window_start       |window_end         |bot  |edit_count|total_bytes|avg_bytes|
|2026-10-01 14:39:30|2026-10-01 14:40:00|false|10        |1991       |199.1    |
|2026-10-01 14:38:00|2026-10-01 14:38:30|false|2         |15         |7.5      |
|2026-10-01 14:39:00|2026-10-01 14:39:30|false|8         |292        |36.5     |
|2026-10-01 14:38:30|2026-10-01 14:39:00|false|10        |-401       |-40.1    |
|2026-10-01 14:38:30|2026-10-01 14:39:00|true |2         |912        |456.0    |
|2026-10-01 14:39:00|2026-10-01 14:39:30|true |4         |1541       |385.25   |
```

Observations. The window bounds are aligned on 14:38:00, 14:38:30, 14:39:00: consecutive slices with no overlap, so the counts can be summed to get the real number of edits.

Bots make far fewer edits but much larger ones: around 385-456 bytes on average, against 7 to 199 for humans. This makes sense, as bots add whole lists, templates or categories at once.

Some windows have a negative total (-401 bytes): during that slice, more text was removed than added.

### Exploration 2 - Overlapping windows (60 seconds every 20 seconds)

"window(col("event_time"), "60 seconds", "20 seconds")" takes a second duration: the slide. A new 60-second window starts every 20 seconds, so the windows overlap and each edit is counted in 3 of them. This smooths the variations, like a moving average.

In [10]:
sliding_df = events_df \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        window(col("event_time"), "60 seconds", "20 seconds")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg(_abs(col("size_change"))).alias("avg_abs_bytes")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("edit_count"),
        col("avg_abs_bytes")
    )

sliding_query = sliding_df \
    .writeStream \
    .queryName("sliding_60s_every_20s") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()
print("hey")

hey


**Output (French Wikipedia)**

```
-------------------------------------------
Batch: 15
-------------------------------------------
|window_start       |window_end         |edit_count|avg_abs_bytes     |
|2026-10-01 14:39:40|2026-10-01 14:40:40|19        |242.57894736842104|
|2026-10-01 14:40:40|2026-10-01 14:41:40|8         |39.375            |
|2026-10-01 14:39:00|2026-10-01 14:40:00|22        |176.45454545454547|
|2026-10-01 14:40:20|2026-10-01 14:41:20|15        |183.46666666666667|
|2026-10-01 14:40:00|2026-10-01 14:41:00|20        |150.8             |
|2026-10-01 14:39:20|2026-10-01 14:40:20|20        |130.5             |
```

Observations. The windows start every 20 seconds (14:39:20, 14:39:40, 14:40:00, 14:40:20) but each lasts 60 seconds, so consecutive rows cover common periods.

| | Tumbling (30s) | Sliding (60s / 20s) |
|---|---|---|
| Window starts | every 30 s | every 20 s |
| Overlap | none | 40 s |
| An edit is counted | once | 3 times |
| Sum of the counts | real number of edits | about 3x the real number |

The last line is the important one: with sliding windows the rows must not be summed, since the same edit appears in several of them. Sliding windows answer "how busy has the last minute been, updated every 20 seconds", while tumbling windows answer "how many edits happened in each 30-second slice".

In [26]:
# stop the previous queries to keep the console readable
tumbling_query.stop()
sliding_query.stop()
print("hey")

hey


## Q10 - Additional aggregations

### Exploration 3 - Most active users (1-minute window)

Grouping by user on top of the window gives a ranking. Note that "orderBy" is only allowed in "complete" output mode, because Spark needs the full state to sort the rows.

In [19]:
from pyspark.sql.functions import max as _max

top_users_df = events_df \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("user"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        _sum(_abs(col("size_change"))).alias("total_bytes_changed")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("user"),
        col("bot"),
        col("edit_count"),
        col("total_bytes_changed")
    ) \
    .orderBy(col("edit_count").desc())

top_users_query = top_users_df \
    .writeStream \
    .queryName("top_users_1min") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .option("numRows", 10) \
    .start()
print("hey")

hey


**Output**

```
-------------------------------------------
Batch: 21
-------------------------------------------
|window_start       |user       |bot  |edit_count|total_bytes_changed|
|2026-10-01 15:02:00|Matpib     |false|6         |102                |
|2026-10-01 15:00:00|Matpib     |false|6         |95                 |
|2026-10-01 14:59:00|Matpib     |false|4         |55                 |
|2026-10-01 15:02:00|Εὐθυμένης  |false|4         |132                |
|2026-10-01 15:02:00|OrlodrimBot|true |3         |767                |
|2026-10-01 15:01:00|CodexBot2  |true |2         |191                |
|2026-10-01 14:59:00|CodexBot   |true |1         |492                |
```

Observations. A few regular contributors stand out (Matpib with 6 edits per minute), while most users appear only once per window: on such a short window the ranking is not very discriminating, a 5-minute window would separate regular contributors better.

The bot / human contrast is clear again: OrlodrimBot changed 767 bytes in 3 edits, and CodexBot 492 bytes in a single edit, against 102 bytes for the 6 edits of `Matpib`. Bots are also easy to spot by their names.

In [13]:
top_users_query.stop()

### Exploration 4 - Edit types and minor edits (30-second window)

Grouping by type (edit, new page, categorize...) and by minor (small fixes such as typos or formatting).

In [20]:
types_df = events_df \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        window(col("event_time"), "30 seconds"),
        col("type"),
        col("minor")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("type"),
        col("minor"),
        col("edit_count")
    )

types_query = types_df \
    .writeStream \
    .queryName("edit_types_30s") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()
print("hey")

hey


**Output**

```
-------------------------------------------
Batch: 8
-------------------------------------------
|window_start       |window_end         |type|minor|edit_count|
|2026-10-01 15:02:00|2026-10-01 15:02:30|edit|true |10        |
|2026-10-01 15:02:30|2026-10-01 15:03:00|edit|true |9         |
|2026-10-01 15:02:30|2026-10-01 15:03:00|edit|false|5         |
|2026-10-01 15:02:00|2026-10-01 15:02:30|edit|false|4         |
|2026-10-01 15:03:00|2026-10-01 15:03:30|edit|false|2         |
|2026-10-01 15:01:30|2026-10-01 15:02:00|edit|true |3         |
```

Observations. Only the type edit appears, never new or categorize. This is not a property of Wikipedia but a consequence of the producer filter type='edit' (see Q11): page creations and log events were already discarded upstream.

Minor edits are the majority (10 against 4, 9 against 5), which is consistent with the nature of the stream: most contributions are small corrections rather than new content.

In [22]:
types_query.stop()

### Exploration 5 - Shorter window (10 seconds)

Same aggregation as before but with a 10-second window, to see the effect of the window length on the results.

In [23]:
short_window_df = events_df \
    .withWatermark("event_time", "30 seconds") \
    .groupBy(window(col("event_time"), "10 seconds")) \
    .agg(
        count("*").alias("edit_count"),
        avg(_abs(col("size_change"))).alias("avg_abs_bytes")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("edit_count"),
        col("avg_abs_bytes")
    )

short_window_query = short_window_df \
    .writeStream \
    .queryName("tumbling_10s") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()
print("hey")

IllegalArgumentException: Cannot start query with name tumbling_10s as a query with that name is already active in this SparkSession

**Output**

```
-------------------------------------------
Batch: 3
-------------------------------------------
|window_start       |window_end         |edit_count|avg_abs_bytes     |
|2026-10-01 15:03:00|2026-10-01 15:03:10|4         |149.5             |
|2026-10-01 15:03:10|2026-10-01 15:03:20|6         |27.666666666666668|
|2026-10-01 15:02:50|2026-10-01 15:03:00|3         |21.666666666666668|
|2026-10-01 15:02:40|2026-10-01 15:02:50|2         |1443.5            |
```

Observations. Short windows react much faster: a new row appears every 10 seconds instead of every 30 seconds or every hour, which shows the irregular rhythm of the stream.

The trade-off is noise: each window holds only 2 to 6 edits, so a single large edit distorts the average. The window 15:02:40-15:02:50 shows 1443.5 bytes on average over 2 edits only, while the neighbouring windows stay between 20 and 150. With a 30-second or 1-minute window this outlier would have been absorbed.

**Window length summary**

| Window | Reactivity | Stability | Use |
|---|---|---|---|
| 10 s | very high | noisy | detect spikes |
| 30 s | good | reasonable | general monitoring |
| 1 min | slower | stable | user rankings |
| 1 hour | very slow | very stable | daily/hourly trends |

In [18]:
short_window_query.stop()

In [24]:
for q in spark.streams.active:
    q.stop()

## Q11 - Filtering at the producer level

The producer already filtered the stream before sending it to Kafka:


stream.register_filter(server_name='fr.wikipedia.org', type='edit')

We changed the language edition to Japanese:


stream.register_filter(server_name='ja.wikipedia.org', type='edit')


Result. With the same 30-second tumbling window, the Japanese stream was clearly slower: 2 to 5 edits per window, against 8 to 10 for the French one, and no bot edit appeared at all during our observation.

```
-------------------------------------------
Batch: 3
-------------------------------------------
|window_start       |window_end         |bot  |edit_count|total_bytes|avg_bytes|
|2026-10-01 15:12:00|2026-10-01 15:12:30|false|5         |-144       |-28.8    |
```

This fits the local time: it was around 2 a.m. in Japan (UTC+9), so human activity is low. Our observation only lasted a few minutes, so this is not a general comparison between the two editions.

**Note on byte counts.** The size_change metric is not comparable across languages: a Japanese character takes 3 bytes in UTF-8 against 1 for most Latin characters, so the same number of characters produces a much larger byte change in Japanese.

**Producer-side vs consumer-side filtering.** Filtering at the producer reduces what goes through Kafka (less network, less storage), but the discarded events are lost: we could not compare French and Japanese afterwards without changing the producer again and waiting. Filtering in Spark keeps everything available for any later analysis, at the cost of a larger stream.